# Week 20: Computer Vision, 3D Vision & NeRFs
## Notebook 07: Segment Anything (SAM): Architecture & Prompt Encoder

### 1. Conceptual Intuition & Theoretical Background
The **Segment Anything Model (SAM)** by Meta AI introduces foundation models to computer vision image segmentation via promptable design.

SAM consists of three decoupled components:
1. **Heavy Image Encoder (ViT):** Runs once per image, mapping high-res images to a $64 \times 64$ feature embedding grid $F_{\text{img}} \in \mathbb{R}^{64 \times 64 \times 256}$.
2. **Prompt Encoder:** Encodes sparse prompts (points, bounding boxes) and dense prompts (masks):
   - Points are mapped via **Positional Fourier Features**:
$$\gamma(v) = [\sin(2\pi B v), \cos(2\pi B v)]$$
   - Plus learnable embeddings indicating foreground vs background points.
3. **Lightweight Mask Decoder:** Uses **Two-Way Cross-Attention** between prompt tokens and image embeddings.
   - Generates 3 multi-mask outputs to resolve ambiguous prompts (e.g. clicking on a backpack: backpack vs person wearing it).
   - Operates in real-time ($<50\text{ms}$) in a web browser via WebAssembly.

---

### 2. Scratch Python Implementation: SAM Sparse Prompt Encoder

In [ ]:
import numpy as np

class SAMPromptEncoderScratch:
    def __init__(self, embed_dim=256, num_fourier_frequencies=128):
        self.embed_dim = embed_dim
        # Gaussian random matrix for Fourier position encoding
        np.random.seed(42)
        self.B_matrix = np.random.normal(0, 1.0, (2, num_fourier_frequencies))
        
        # Learnable prompt type embeddings
        self.fg_point_embed = np.random.normal(0, 0.1, embed_dim)
        self.bg_point_embed = np.random.normal(0, 0.1, embed_dim)
        
    def _fourier_encode_coordinates(self, coords):
        """
        coords: Normalized (x, y) coordinates in [0, 1] of shape (N, 2)
        """
        # Project coordinates: (N, 2) @ (2, F) -> (N, F)
        proj = 2.0 * np.pi * np.dot(coords, self.B_matrix)
        # Concatenate sin and cos
        return np.concatenate([np.sin(proj), np.cos(proj)], axis=-1)
        
    def encode_points(self, point_coords, point_labels):
        """
        point_coords: (N, 2) normalized points
        point_labels: 1 for foreground, 0 for background
        """
        pos_embeddings = self._fourier_encode_coordinates(point_coords)
        
        prompt_tokens = []
        for i in range(len(point_coords)):
            token = pos_embeddings[i].copy()
            if point_labels[i] == 1:
                token += self.fg_point_embed
            else:
                token += self.bg_point_embed
            prompt_tokens.append(token)
            
        return np.array(prompt_tokens)

# Encode 2 user click prompts:
# Point 1: (0.45, 0.55) -> Foreground Click (Label 1)
# Point 2: (0.10, 0.10) -> Background Click (Label 0)
clicks = np.array([[0.45, 0.55], [0.10, 0.10]])
labels = np.array([1, 0])

encoder = SAMPromptEncoderScratch(embed_dim=256)
prompt_tokens = encoder.encode_points(clicks, labels)

# Lightweight Two-Way Decoder Simulation: Dot product with image grid
image_embedding_grid = np.random.randn(64, 64, 256)
# Flatten grid: (4096, 256)
flat_image = image_embedding_grid.reshape(-1, 256)

# Cross-similarity between prompt token 0 and image grid
similarity_map = np.dot(flat_image, prompt_tokens[0]).reshape(64, 64)
predicted_mask = similarity_map > np.percentile(similarity_map, 90)

print("--- Segment Anything (SAM) Prompt Encoding ---")
print(f"Encoded Prompt Tokens Shape:  {prompt_tokens.shape}")
print(f"Image Embedding Grid Shape:   {image_embedding_grid.shape}")
print(f"Segmented Mask Pixel Count:   {np.sum(predicted_mask)} / 4096 pixels")
print("Prompt tokens successfully modulate segmentation mask activation!")

### 3. Step-by-Step Code Walkthrough
1. **Fourier Positional Encoding**: Maps 2D coordinate points into high-dimensional geometric features.
2. **Semantic Type Embedding**: Differentiates foreground clicks from background exclusion clicks.
3. **Decoupled Architecture**: Enables interactive browser inference because the heavy ViT encoder runs only once.

---

### 4. Industry Applications & Trade-offs
- **Medical Imaging & Robotics:** Zero-shot surgical tool segmentation and interactive labeling workflows.
- **Trade-off:** Heavy ViT-H encoder requires significant GPU VRAM during the initial image feature extraction pass.

---

### 5. Top Interview Questions & Answers
**Q1: Why does SAM separate the heavy image encoder from the lightweight mask decoder?**
> **Answer:** The heavy ViT image encoder computes the image embedding once. The lightweight prompt decoder ($<50\text{ms}$) can then execute repeatedly in real-time as the user moves their mouse or clicks points.

**Q2: How does SAM handle prompt ambiguity (e.g. clicking on an object within an object)?**
> **Answer:** It outputs 3 distinct masks corresponding to whole, part, and subpart granularities, along with predicted IoU confidence scores, allowing the user or ranking system to select the correct mask.